# 03 — Training results (walk-forward OOF)

Milestone C. Inspects the 48-cell (sector x horizon) XGBoost training results: out-of-fold regression R², classification accuracy, edge over naive baseline, predicted-vs-actual scatter for the best cells.

**Honest baseline**: monthly return prediction is notoriously hard. Most cells have negative OOF R² (model worse than predicting the mean) and accuracy near the naive majority-class baseline. The few cells that do show edge (e.g. Hlth h=12, Other h=1) are the interesting ones — SHAP in milestone D will explain *why*.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent
metrics = pd.read_csv(ROOT / 'outputs/report/tables/oof_metrics.csv')
oof_reg = pd.read_parquet(ROOT / 'outputs/oof/oof_reg_v1.parquet')
oof_clf = pd.read_parquet(ROOT / 'outputs/oof/oof_clf_v1.parquet')
Y = pd.read_parquet(ROOT / 'outputs/targets/Y_v1.parquet')

metrics.shape, oof_reg.shape

## Regression OOF R² heatmap

In [ ]:
reg = metrics[metrics['task']=='reg']
r2_pivot = reg.pivot(index='sector', columns='horizon', values='r2')

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(r2_pivot, cmap='RdBu', center=0, vmin=-1.0, vmax=0.05,
            annot=True, fmt='.3f', ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Out-of-fold regression R² (sector × horizon)')
ax.set_ylabel('Sector'); ax.set_xlabel('Horizon (months)')
plt.tight_layout()
plt.show()

## Classification edge over naive majority-class baseline

Edge = accuracy − max(base_rate, 1−base_rate). Positive values beat always-predict-same-class.

In [ ]:
clf = metrics[metrics['task']=='clf'].copy()
clf['naive_baseline'] = clf['base_rate'].combine(1-clf['base_rate'], max)
clf['edge'] = clf['accuracy'] - clf['naive_baseline']
edge_pivot = clf.pivot(index='sector', columns='horizon', values='edge')

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(edge_pivot, cmap='RdBu_r', center=0, vmin=-0.2, vmax=0.05,
            annot=True, fmt='+.3f', ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Classification edge vs naive baseline (sector × horizon)')
ax.set_ylabel('Sector'); ax.set_xlabel('Horizon (months)')
plt.tight_layout()
plt.show()

## Best cells (top-10 by edge)

In [ ]:
top = clf.sort_values('edge', ascending=False).head(10)
top[['sector','horizon','accuracy','naive_baseline','edge','f1']].round(3)

## Predicted vs actual — best regression cell

In [ ]:
# Best cell by R² (with positive R² if any)
best = reg.sort_values('r2', ascending=False).iloc[0]
s, h = best['sector'], int(best['horizon'])
print(f'Best regression cell: {s} h={h} R²={best["r2"]:.4f}')

pred = oof_reg[(s, h)].dropna()
actual = Y[(s, h)].reindex(pred.index)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(actual, pred, s=10, alpha=0.5, color='#1f3b66')
lim = max(abs(actual.min()), abs(actual.max()))
axes[0].plot([-lim, lim], [-lim, lim], 'r--', lw=0.6)
axes[0].set_xlabel('Actual CAR'); axes[0].set_ylabel('Predicted CAR')
axes[0].set_title(f'{s} h={h} — predicted vs actual')
axes[0].grid(alpha=0.3)

axes[1].plot(actual.index, actual, label='actual', lw=0.8, color='black')
axes[1].plot(pred.index, pred, label='OOF pred', lw=0.8, color='#c94a3a', alpha=0.8)
axes[1].axhline(0, color='gray', lw=0.3)
axes[1].set_title(f'{s} h={h} CAR over time')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Hlth h=12 — the standout classification cell

57.3% accuracy vs 53.2% naive baseline = +4.1pt edge over 370 OOF predictions.

In [ ]:
pred_clf = oof_clf[('Hlth', 12)].dropna().astype(int)
actual_sign = (Y[('Hlth', 12)].reindex(pred_clf.index) > 0).astype(int)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(actual_sign, pred_clf)
print('Confusion matrix (rows=actual, cols=predicted):')
print(pd.DataFrame(cm, index=['actual_down','actual_up'], columns=['pred_down','pred_up']))
print(f'\nActual positive rate: {actual_sign.mean():.3f}')
print(f'Predicted positive rate: {pred_clf.mean():.3f}')
print(f'Accuracy: {(actual_sign==pred_clf).mean():.3f}')